In [5]:
!pip install transformers datasets accelerate evaluate torch scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [6]:
from google.colab import files

uploaded = files.upload()

Saving Fake.csv to Fake.csv
Saving True.csv to True.csv


In [7]:
import pandas as pd

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

print(fake.shape)
print(true.shape)

(23481, 4)
(21417, 4)


In [8]:
fake["label"] = 0
true["label"] = 1

In [9]:
df = pd.concat([fake, true])

df = df[["text", "label"]]

df = df.sample(frac=1, random_state=42)

df.reset_index(drop=True, inplace=True)

df.head()

,text,label
0,"21st Century Wire says Ben Stein, reputable pr...",0
1,WASHINGTON (Reuters) - U.S. President Donald T...,1
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,1
3,"On Monday, Donald Trump once again embarrassed...",0
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",1


In [10]:
df = df.sample(10000, random_state=42)

In [11]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [12]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [13]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [14]:
import torch

class NewsDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):
        return len(self.labels)

In [15]:
train_dataset = NewsDataset(
    train_encodings,
    train_labels
)

test_dataset = NewsDataset(
    test_encodings,
    test_labels
)

In [16]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch"
)

In [18]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.004464,0.005301
2,0.000052,0.000038


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=0.010829752626828849, metrics={'train_runtime': 429.9863, 'train_samples_per_second': 37.21, 'train_steps_per_second': 4.651, 'total_flos': 1059739189248000.0, 'train_loss': 0.010829752626828849, 'epoch': 2.0})

In [21]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
0.000052,0.000038,2


{'eval_loss': 3.833310984191485e-05}

In [22]:
model.save_pretrained("fake_news_model")
tokenizer.save_pretrained("fake_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('fake_news_model/tokenizer_config.json', 'fake_news_model/tokenizer.json')

In [24]:
from transformers import pipeline

# Initialize the pipeline with your trained model and tokenizer
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

text = """
The government announced a new
economic policy today.
"""

result = classifier(text)

print(result)

[{'label': 'LABEL_0', 'score': 0.999903678894043}]


In [25]:
label_map = {
    "LABEL_0":"FAKE",
    "LABEL_1":"REAL"
}

news = input("Enter News:\n")

result = classifier(news)[0]

print()

print(
    "Prediction:",
    label_map[result["label"]]
)

print(
    "Confidence:",
    round(result["score"]*100,2),
    "%"
)

Enter News:
four killed in texas

Prediction: FAKE
Confidence: 99.99 %
